In [1]:
# Connect to Cassandra (container: my_cassandra)
from cassandra.cluster import Cluster

cluster = Cluster(['localhost'], port=9042)  # Same port as opened in 'docker run -p 9042:9042'
session = cluster.connect()                   # A session is used to send CQL queries


In [2]:
# Create a keyspace for the project (first time only)
# SimpleStrategy + replication_factor 1: the data is stored once, enough for one local node
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS ind320
    WITH REPLICATION = {'class': 'SimpleStrategy', 'replication_factor': 1};
""")

In [3]:
# List all keyspaces to confirm that 'ind320' was created
rows = session.execute("SELECT keyspace_name FROM system_schema.keyspaces;")
for row in rows:
    print(row.keyspace_name)

ind320
system_auth
system_schema
system_distributed
system
system_traces


In [4]:
# Environment variables for PySpark (system dependent)
import os, sys

# Java 17 (Microsoft OpenJDK) – path without \bin\java.exe
os.environ["JAVA_HOME"] = r"C:\Program Files\Microsoft\jdk-17.0.20.101-hotspot"
# Make Spark use the same Python as this notebook (.venv), not the system Python
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [5]:
# Start a Spark session with the Spark-Cassandra connector
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('IND320_part2').\
    config('spark.jars.packages', 'com.datastax.spark:spark-cassandra-connector_2.12:3.5.1').\
    config('spark.cassandra.connection.host', 'localhost').\
    config('spark.cassandra.connection.port', '9042').\
    config('spark.sql.extensions', 'com.datastax.spark.connector.CassandraSparkExtensions').\
    config('spark.sql.catalog.mycatalog', 'com.datastax.spark.connector.datasource.CassandraCatalog').\
    getOrCreate()

In [6]:
# Create a small test table in Cassandra and insert one row
session.execute("CREATE TABLE IF NOT EXISTS ind320.test (id int PRIMARY KEY, name text);")
session.execute("INSERT INTO ind320.test (id, name) VALUES (1, 'hello spark');")

# Read the same table through Spark
spark.read.format("org.apache.spark.sql.cassandra").\
    options(table="test", keyspace="ind320").load().show()

+---+-----------+
| id|       name|
+---+-----------+
|  1|hello spark|
+---+-----------+

